In [2]:
import os
import pandas as pd
import mysql.connector

# -----------------------------
# MySQL Configuration
# -----------------------------
HOST = "localhost"
USER = "root"
PASSWORD = "@201517#aa"
DATABASE = "ecommerce"

# -----------------------------
# Dataset Folder
# -----------------------------
FOLDER_PATH = r"C:\Users\dk523\Documents\projects\E-Commerce-Sales-Analytics\dataset"

# CSV File -> Table Name
csv_files = [
    ("customers.csv", "customers"),
    ("orders.csv", "orders"),
    ("sellers.csv", "sellers"),
    ("products.csv", "products"),
    ("geolocation.csv", "geolocation"),
    ("payments.csv", "payments"),
    ("order_items.csv", "order_items")
]

# -----------------------------
# Create Connection
# -----------------------------
conn = mysql.connector.connect(
    host=HOST,
    user=USER,
    password=PASSWORD
)

cursor = conn.cursor()

# Create database if not exists
cursor.execute(f"CREATE DATABASE IF NOT EXISTS {DATABASE}")
cursor.execute(f"USE {DATABASE}")

print(f"Connected to database: {DATABASE}")

# -----------------------------
# SQL Datatype Mapper
# -----------------------------
def get_sql_type(dtype):

    if pd.api.types.is_integer_dtype(dtype):
        return "BIGINT"

    elif pd.api.types.is_float_dtype(dtype):
        return "DOUBLE"

    elif pd.api.types.is_bool_dtype(dtype):
        return "BOOLEAN"

    elif pd.api.types.is_datetime64_any_dtype(dtype):
        return "DATETIME"

    return "TEXT"

# -----------------------------
# Import CSV Files
# -----------------------------
for csv_file, table_name in csv_files:

    print(f"\nProcessing {csv_file}")

    file_path = os.path.join(FOLDER_PATH, csv_file)

    df = pd.read_csv(file_path)

    df.columns = (
        df.columns.str.strip()
                  .str.replace(" ", "_")
                  .str.replace("-", "_")
                  .str.replace(".", "_", regex=False)
    )

    df = df.where(pd.notnull(df), None)

    # Drop table if exists
    cursor.execute(f"DROP TABLE IF EXISTS `{table_name}`")

    # Create Table
    columns = ", ".join(
        [f"`{col}` {get_sql_type(df[col].dtype)}"
         for col in df.columns]
    )

    create_query = f"""
    CREATE TABLE `{table_name}` (
        {columns}
    )
    """

    cursor.execute(create_query)

    # Bulk Insert (FAST)
    placeholders = ", ".join(["%s"] * len(df.columns))
    column_names = ", ".join([f"`{c}`" for c in df.columns])

    insert_query = f"""
    INSERT INTO `{table_name}`
    ({column_names})
    VALUES ({placeholders})
    """

    data = [
        tuple(None if pd.isna(x) else x for x in row)
        for row in df.itertuples(index=False, name=None)
    ]

    BATCH_SIZE = 5000

    for i in range(0, len(data), BATCH_SIZE):
        cursor.executemany(insert_query, data[i:i+BATCH_SIZE])
        conn.commit()

    print(f"✔ Imported {len(df):,} rows into {table_name}")

cursor.close()
conn.close()

print("\n=================================")
print("All tables imported successfully.")
print("=================================")

Connected to database: ecommerce

Processing customers.csv
✔ Imported 99,441 rows into customers

Processing orders.csv
✔ Imported 99,441 rows into orders

Processing sellers.csv
✔ Imported 3,095 rows into sellers

Processing products.csv
✔ Imported 32,951 rows into products

Processing geolocation.csv
✔ Imported 1,000,163 rows into geolocation

Processing payments.csv
✔ Imported 103,886 rows into payments

Processing order_items.csv
✔ Imported 112,650 rows into order_items

All tables imported successfully.


In [3]:
!pip install mysql-connector-python


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import mysql.connector
import pandas as pd

conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="@201517#aa",
    database="ecommerce"
)

tables = [
    "customers",
    "orders",
    "products",
    "payments",
    "order_items",
    "sellers",
    "geolocation"
]

for table in tables:

    query = f"SELECT COUNT(*) FROM {table}"

    count = pd.read_sql(query, conn).iloc[0,0]

    print(f"{table:<15} : {count:,}")

conn.close()

C:\Users\dk523\AppData\Local\Temp\ipykernel_18156\3000448560.py:25: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  count = pd.read_sql(query, conn).iloc[0,0]


customers       : 99,441
orders          : 99,441
products        : 32,951
payments        : 103,886
order_items     : 112,650
sellers         : 3,095
geolocation     : 1,000,163
